# Протонный диод В. Лифшица: ЭДС ХХ, ток КЗ и ВАХ по схеме Нернста–Планка
*16 ноя 2026. Постановка — по письму В. Лифшица от 16.11.2026 и с. 4 материала `Hydrogen_Energetics_NP.pdf`.*

**Схема (одна общая газовая фаза H₂, давление p, температура T):**
H₂(газ) ⇄ 2H(вход, Pt) ⇄ H⁺ + e⁻ → **мембрана** (диффузия ↔ дрейф) → H⁺ + e⁻ ⇄ H(выход, Pt) ⇄ **барьерный слой** ⇄ H₂(тот же газ).

**Как предложил Владимир:**
- ЭДС определяется активностями **атомарного H в каталитических слоях** a_вх и a_вых, а не давлениями газа;
- поток протонов в мембране = диффузия (вход→выход) минус дрейф (обратно, против поля); в ХХ они равны;
- I_КЗ = e·(поток протонов) при эквипотенциальных электродах.

**Единственный неизвестный — барьер.** Вводим параметр κ: во сколько раз абсорбция H₂ барьером (H₂ → 2H) подавлена сильнее, чем требует равновесие H₂ ⇄ 2H.
- κ = 1 — микрообратимость (катализатор/барьер меняет скорость, но не равновесие);
- κ > 1 — гипотеза гомофлюидного осмоса («выход есть, входа почти нет»).

Все скорости — закон действующих масс; потенциалы в единицах RT/F; константы — произвольные.

In [ ]:
# preparser(False)  # раскомментировать при запуске в SageMath 9.6

In [ ]:
import numpy as np
RT_F_mV = 8.314462618*300/96485.33212*1e3   # 25.85 мВ при 300 K

def rates(x, P, kappa=1.0, p=1.0):
    """x = [ln a_in, u0, uL, ln a_out, u2]; потенциалы в единицах RT/F; phi1 = 0.
    a_in, a_out — активности атомарного H во входном и выходном каталитических слоях,
    u0, uL — потенциал протонов у входной/выходной границы мембраны (c = 1, Nafion),
    u2 — потенциал выходного электрода. kappa = 1: барьер с микрообратимостью;
    kappa > 1: абсорбция H2 барьером подавлена в kappa раз сильнее, чем «положено» (гипотеза).
    Все потоки — в атомах H (= протонах = электронах) за единицу времени."""
    la, u0, uL, lo, u2 = x
    a, ao = np.exp(la), np.exp(lo)
    r1 = 2*P['k1']*(p - a**2)                                   # H2(газ) ⇄ 2H(вход)
    d1 = u0 - 0.0                                               # перенапряжение входа
    r2 = P['k2']*(a*np.exp(-d1/2) - np.exp(d1/2))               # H(вход) ⇄ H+ + e(эл.1)
    J  = P['g']*(np.exp(u0) - np.exp(uL))                       # мембрана: диффузия + дрейф (Нернст–Планк)
    d2 = uL - u2
    r4 = P['k4']*(np.exp(d2/2) - ao*np.exp(-d2/2))              # H+ + e(эл.2) ⇄ H(выход)
    r5 = 2*P['k5']*(ao**2 - p/kappa)                            # 2H(выход, барьер) ⇄ H2(газ)
    return r1, r2, J, r4, r5

def solve(P, kappa=1.0, mode='load', R=0.0, V=None, x0=None):
    """mode='load': внешняя нагрузка R (R=0 — КЗ, R=inf — ХХ); mode='V': задано V = phi2 - phi1."""
    def F(x):
        r1, r2, J, r4, r5 = rates(x, P, kappa)
        I = r4
        if mode == 'V':   last = x[4] - V
        elif np.isinf(R): last = I                              # ХХ: ток во внешней цепи = 0
        else:             last = x[4] - (-I*R)                  # phi2 = -I*R (ток I течёт 2→1 снаружи)
        return np.array([r1-r2, r2-J, J-r4, r4-r5, last])
    x = np.zeros(5) if x0 is None else np.array(x0, float)
    for _ in range(200):
        f = F(x)
        Jm = np.empty((5,5)); h = 1e-7
        for j in range(5):
            dx = np.zeros(5); dx[j] = h
            Jm[:,j] = (F(x+dx)-f)/h
        step = np.linalg.solve(Jm, -f)
        x += np.clip(step, -2, 2)
        if np.max(abs(step)) < 1e-13: break
    I = rates(x, P, kappa)[3]
    return x, I


## 1. Барьер с микрообратимостью (κ = 1): 30 случайных наборов констант (разброс 10⁻³…10³)

In [ ]:
rng = np.random.default_rng(1)
worst_V, worst_I = 0.0, 0.0
for _ in range(30):
    P = {k: 10**rng.uniform(-3, 3) for k in ['k1','k2','g','k4','k5']}
    x, _  = solve(P, 1.0, 'load', R=np.inf); Voc = x[4]*RT_F_mV
    x2, Isc = solve(P, 1.0, 'load', R=0.0)
    a_in, a_out = np.exp(x[0]), np.exp(x[3])
    worst_V, worst_I = max(worst_V, abs(Voc)), max(worst_I, abs(Isc))
print('max |E_ХХ| =', worst_V, 'мВ;  max |I_КЗ| =', worst_I)
print('OK: E_ХХ = I_КЗ = 0 при любых константах' if worst_V < 1e-6 and worst_I < 1e-10 else 'FAIL')

max |E_ХХ| = 0.0 мВ;  max |I_КЗ| = 0.0
OK: E_ХХ = I_КЗ = 0 при любых константах


**Почему ноль.** В ХХ ток не течёт ни через мембрану, ни через барьер. Нет потока через барьер → a_вых равно равновесному с газом; нет потока через вход → a_вх тоже равновесно с тем же газом. Значит a_вых = a_вх, и формула Нернста в форме Владимира, E = (RT/F)·ln(a_вх/a_вых), даёт 0. «Откачка» выходного слоя барьером в стационаре прекращается, если барьер подчиняется микрообратимости.

## 2. Гипотеза гомофлюидного осмоса: κ > 1

In [ ]:
P = {'k1':1, 'k2':5, 'g':2, 'k4':0.05, 'k5':0.2}
print(' kappa   E_XX, мВ   (RT/2F)ln(kappa)   I_КЗ (отн.)')
for kap in [1, 1.08, 2, 10, 100, 1e4]:
    x, _ = solve(P, kap, 'load', R=np.inf); E = x[4]*RT_F_mV
    _, Isc = solve(P, kap, 'load', R=0.0)
    print(f'{kap:7g}  {E:9.4f}   {RT_F_mV/2*np.log(kap):9.4f}        {Isc:.4f}')

 kappa   E_XX, мВ   (RT/2F)ln(kappa)   I_КЗ (отн.)
      1     0.0000      0.0000        0.0000
   1.08     0.9948      0.9948        0.0017
      2     8.9596      8.9596        0.0130
     10    29.7632     29.7632        0.0284
    100    59.5264     59.5264        0.0336
  10000   119.0529    119.0529        0.0343


**Результат:** E_ХХ = (RT/2F)·ln κ **точно**, независимо от констант мембраны и электродов. То есть ЭДС измеряет ровно одну величину — асимметрию барьера κ:
- 1 мВ ⇔ κ ≈ 1.08; 10 мВ ⇔ κ ≈ 2.2; 60 мВ ⇔ κ = 100.

I_КЗ, напротив, зависит от кинетики и ограничен самой медленной стадией (здесь k4 — выходной электрод).

Важно: κ ≠ 1 — это допущение, что стадия 2H ⇄ H₂ на барьере имеет отношение прямой и обратной скоростей, отличное от exp(−ΔG/RT). Модель показывает, **что** даст такое допущение, но не обосновывает его.

## 3. ВАХ при внешнем напряжении (потенциостат), κ = 1

In [ ]:
P_asym = {'k1':1, 'k2':5, 'g':2, 'k4':0.05, 'k5':0.2}   # сильно разные электроды
print('  V, мВ      I (отн.)')
for v in np.linspace(-6, 6, 13):
    _, I = solve(P_asym, 1.0, 'V', V=v)
    print(f'{v*RT_F_mV:8.1f}   {I:+.4f}')
_, I0 = solve(P_asym, 1.0, 'V', V=0.0)
print('I(V=0) =', I0, '->', 'OK: ВАХ проходит через 0' if abs(I0) < 1e-12 else 'FAIL')

  V, мВ      I (отн.)
  -155.1   +0.6319
  -129.3   +0.4528
  -103.4   +0.3017
   -77.6   +0.1882
   -51.7   +0.1070
   -25.9   +0.0477
     0.0   +0.0000
    25.9   -0.0451
    51.7   -0.0946
    77.6   -0.1527
   103.4   -0.2188
   129.3   -0.2850
   155.1   -0.3387
I(V=0) = 0.0 -> OK: ВАХ проходит через 0


ВАХ **асимметрична** (при ±155 мВ ток +0.63 и −0.34): это обычное выпрямление из-за разной кинетики электродов. Но она проходит **через начало координат**. Асимметрия ВАХ сама по себе генерацию не доказывает; решают только две точки: **I(V=0)** и **V(I=0)**.

## 4. Протокол для потенциостата
1. Одна камера H₂ (обе стороны в одном газе), изотермия ΔT < 0.01 K, O₂ < 1 ppm.
2. Измерить V(I=0) электрометром (≥10¹² Ом) часы—сутки и I(V=0) — 0 В на потенциостате.
3. Снять ВАХ ±200 мВ; проверить, проходит ли через (0, 0).
4. Перевернуть ячейку: эффект барьера обязан сменить знак.
5. Контроль: та же ячейка без барьерного слоя (или с барьером с обеих сторон) — эффект должен исчезнуть.
6. Калибровка: разделить камеры, p_вых/p_вх = 1.01 → ожидается 0.13 мВ (Нернст).

**Пересчёт:** измеренная E_ХХ = ε мВ ⇒ κ = exp(2ε/25.85). Отличие κ от 1 выше погрешности (с контролями 4–5) было бы прямым экспериментальным свидетельством нарушения микрообратимости на барьере.